In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!nvidia-smi
!ls /kaggle/input

In [24]:
%%writefile brain_tumor_preprocessing_pipeline.py
"""
Brain Tumor MRI Classification - Data Collection and Preprocessing Module
=========================================================================
One file containing the complete pipeline for the IA 1 review.

Tasks covered
  1. Dataset collection and organization      collect_images(), export_split_folders()
  2. Dataset exploration and statistics       explore()
  3. Data cleaning and quality verification   inspect_dataset(), clean_dataset()
  4. Image preprocessing and augmentation     build_transforms()
  5. Splitting 70% / 15% / 15%                split_dataset(), verify_splits()
  6. PyTorch data-loading pipeline            BrainMRIDataset, build_dataloaders()
  7. Integration and testing                  run_pipeline(), run_tests()

Rules followed
  - The original dataset is only READ, never modified (checked by a fingerprint).
  - Duplicates are removed BEFORE splitting, so no image can sit in two splits.
  - Augmentation is applied to the training set only.

Usage
  pip install torch torchvision pillow pandas numpy scikit-learn matplotlib
  python brain_tumor_preprocessing_pipeline.py --data "C:/path/to/dataset" --out outputs
  python brain_tumor_preprocessing_pipeline.py --test          # runs the 12 test cases

The dataset folder must contain Training/ and Testing/ with one sub-folder per class
(glioma, meningioma, notumor, pituitary). Both parts are pooled and re-split 70/15/15.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import logging
import random
import shutil
import sys
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

log = logging.getLogger("pipeline")

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
IMAGENET_MEAN = (0.485, 0.456, 0.406)   # statistics the pretrained backbones expect
IMAGENET_STD = (0.229, 0.224, 0.225)


@dataclass
class Config:
    data_root: Path
    out_dir: Path = Path("outputs")
    img_size: int = 224
    seed: int = 42
    train_frac: float = 0.70
    val_frac: float = 0.15
    test_frac: float = 0.15
    batch_size: int = 32
    num_workers: int = 0      # 0 is the safest on Windows; raise it on Linux / Kaggle
    min_side: int = 32        # images smaller than this (in width or height) are rejected


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ----------------------------------------------------------------------------
# Task 1: collection and organization
# ----------------------------------------------------------------------------
def collect_images(root: Path) -> pd.DataFrame:
    """Scan the dataset folder (read-only) and list every image with its class."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset folder not found: {root}")
    rows = []
    for p in sorted(root.rglob("*")):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:
            continue
        parts = [x.lower() for x in p.relative_to(root).parts]
        original = "Training" if "training" in parts else "Testing" if "testing" in parts else "Unknown"
        rows.append({"path": str(p), "label": p.parent.name.lower(), "original_split": original})
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No images found under {root}. Point --data to the folder that contains Training and Testing.")
    unknown = sorted(set(df.label) - set(CLASS_NAMES))
    if unknown:
        log.warning("Ignoring folders that are not one of the 4 classes: %s", unknown)
    df = df[df.label.isin(CLASS_NAMES)].reset_index(drop=True)
    log.info("Collected %d images in %d classes", len(df), df.label.nunique())
    return df


def dataset_fingerprint(root: Path) -> str:
    """Hash of (relative path, size) for every file, used to prove the original is untouched."""
    root = Path(root)
    h = hashlib.md5()
    for p in sorted(root.rglob("*")):
        if p.is_file():
            h.update(f"{p.relative_to(root).as_posix()}|{p.stat().st_size}".encode())
    return h.hexdigest()


def export_split_folders(splits: Dict[str, pd.DataFrame], out_dir: Path) -> None:
    """Optional: copy images into organized/<split>/<class>/ (the original stays untouched)."""
    for name, part in splits.items():
        for _, r in part.iterrows():
            src = Path(r.path)
            dst_dir = Path(out_dir) / "organized" / name / r.label
            dst_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst_dir / f"{src.stem}_{r.md5[:8]}{src.suffix.lower()}")
    log.info("Organized copy written to %s", Path(out_dir) / "organized")


# ----------------------------------------------------------------------------
# Task 3 (part 1): inspect every file
# ----------------------------------------------------------------------------
def inspect_image(path: str) -> dict:
    """Open one image fully; report size, mode and a content hash, or the error."""
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im.load()                      # a truncated file fails here
            width, height, mode = im.width, im.height, im.mode
        digest = hashlib.md5(Path(path).read_bytes()).hexdigest()
        return {"ok": True, "width": width, "height": height, "mode": mode, "md5": digest, "error": ""}
    except Exception as exc:               # any read problem means the file is unusable
        return {"ok": False, "width": 0, "height": 0, "mode": "", "md5": "", "error": type(exc).__name__}


def inspect_dataset(df: pd.DataFrame) -> pd.DataFrame:
    info = []
    for i, p in enumerate(df.path, 1):
        info.append(inspect_image(p))
        if i % 1000 == 0:
            log.info("Inspected %d / %d images", i, len(df))
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(info)], axis=1)


# ----------------------------------------------------------------------------
# Task 2: exploration and statistics
# ----------------------------------------------------------------------------
def explore(df: pd.DataFrame, out_dir: Path, plots: bool = True) -> dict:
    good = df[df.ok].copy()
    good["res"] = good.width.astype(str) + "x" + good.height.astype(str)
    stats = {
        "total_images": int(len(df)),
        "readable_images": int(len(good)),
        "class_counts": good.label.value_counts().to_dict(),
        "counts_by_original_split": good.groupby(["original_split", "label"]).size().unstack(fill_value=0).to_dict(),
        "unique_resolutions": int(good.res.nunique()),
        "top_resolutions": good.res.value_counts().head(10).to_dict(),
        "width_min_median_max": [int(good.width.min()), int(good.width.median()), int(good.width.max())],
        "height_min_median_max": [int(good.height.min()), int(good.height.median()), int(good.height.max())],
        "color_modes": good["mode"].value_counts().to_dict(),
    }
    sample = good.sample(min(300, len(good)), random_state=0)       # brightness on a sample only
    means = [np.asarray(Image.open(p).convert("L"), dtype=np.float32).mean() for p in sample.path]
    stats["mean_brightness_0_255"] = round(float(np.mean(means)), 2)
    stats["std_of_brightness_across_images"] = round(float(np.std(means)), 2)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "analysis_report.json").write_text(json.dumps(stats, indent=2))
    if plots:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        good.groupby(["label", "original_split"]).size().unstack(fill_value=0).plot(kind="bar", figsize=(7, 4), title="Images per class (original split)")
        plt.ylabel("Count"); plt.tight_layout(); plt.savefig(out_dir / "class_distribution.png", dpi=150); plt.close()
        good.res.value_counts().head(10).plot(kind="bar", figsize=(7, 4), title="Top 10 image resolutions")
        plt.ylabel("Count"); plt.tight_layout(); plt.savefig(out_dir / "resolutions.png", dpi=150); plt.close()
    log.info("Exploration done: %d unique resolutions, modes %s", stats["unique_resolutions"], stats["color_modes"])
    return stats


# ----------------------------------------------------------------------------
# Task 3 (part 2): cleaning
# ----------------------------------------------------------------------------
def clean_dataset(df: pd.DataFrame, min_side: int) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Drop corrupt, tiny, label-conflicting and duplicate images. Returns (clean, removed, report)."""
    removed = []
    corrupt = df[~df.ok]
    removed.append(corrupt.assign(reason="corrupt_or_unreadable"))
    good = df[df.ok]
    tiny = good[(good.width < min_side) | (good.height < min_side)]
    removed.append(tiny.assign(reason="too_small"))
    good = good.drop(tiny.index)
    n_labels = good.groupby("md5").label.nunique()
    conflict_hashes = n_labels[n_labels > 1].index
    conflicts = good[good.md5.isin(conflict_hashes)]
    removed.append(conflicts.assign(reason="same_image_different_labels"))
    good = good[~good.md5.isin(conflict_hashes)]
    dup_mask = good.duplicated("md5", keep="first")                 # keep the first copy of each image
    removed.append(good[dup_mask].assign(reason="exact_duplicate"))
    good = good[~dup_mask].reset_index(drop=True)
    removed_df = pd.concat(removed, ignore_index=True)
    report = {
        "images_before_cleaning": int(len(df)),
        "corrupt_or_unreadable": int(len(corrupt)),
        "too_small": int(len(tiny)),
        "label_conflicts": int(len(conflicts)),
        "exact_duplicates_removed": int(dup_mask.sum()),
        "images_after_cleaning": int(len(good)),
        "class_counts_after_cleaning": good.label.value_counts().to_dict(),
    }
    log.info("Cleaning: %s", report)
    return good, removed_df, report


# ----------------------------------------------------------------------------
# Task 5: splitting (70 / 15 / 15, stratified, no leakage)
# ----------------------------------------------------------------------------
def split_dataset(df: pd.DataFrame, cfg: Config) -> Dict[str, pd.DataFrame]:
    if abs(cfg.train_frac + cfg.val_frac + cfg.test_frac - 1.0) > 1e-9:
        raise ValueError("train/val/test fractions must add up to 1")
    train, rest = train_test_split(df, test_size=1 - cfg.train_frac, stratify=df.label, random_state=cfg.seed)
    val, test = train_test_split(rest, test_size=cfg.test_frac / (cfg.val_frac + cfg.test_frac), stratify=rest.label, random_state=cfg.seed)
    return {"train": train.reset_index(drop=True), "val": val.reset_index(drop=True), "test": test.reset_index(drop=True)}


def verify_splits(splits: Dict[str, pd.DataFrame]) -> None:
    """Raise if any file path or image content appears in more than one split."""
    names = list(splits)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = splits[names[i]], splits[names[j]]
            if set(a.path) & set(b.path) or set(a.md5) & set(b.md5):
                raise AssertionError(f"Data leakage between {names[i]} and {names[j]}")


# ----------------------------------------------------------------------------
# Task 4: preprocessing and augmentation
# ----------------------------------------------------------------------------
class ToRGB:
    """Make every image 3-channel (handles grayscale, RGBA and palette images)."""
    def __call__(self, img: Image.Image) -> Image.Image:
        return img.convert("RGB")


class PadToSquare:
    """Pad with black to a square so resizing does not stretch the brain."""
    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        if w == h:
            return img
        side = max(w, h)
        left, top = (side - w) // 2, (side - h) // 2
        return ImageOps.expand(img, border=(left, top, side - w - left, side - h - top), fill=0)


def build_transforms(img_size: int, train: bool) -> T.Compose:
    steps = [ToRGB(), PadToSquare(), T.Resize((img_size, img_size))]
    if train:                              # augmentation: training data only, kept mild for MRI
        steps += [
            T.RandomHorizontalFlip(0.5),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05), fill=0),
            T.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    steps += [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return T.Compose(steps)


def denormalize(x: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1)


def save_augmentation_preview(train_df: pd.DataFrame, cfg: Config) -> None:
    """Figure for the demo: each row is one original image followed by 3 augmented versions."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    aug = build_transforms(cfg.img_size, train=True)
    plain = build_transforms(cfg.img_size, train=False)
    picks = [train_df[train_df.label == c].iloc[0] for c in CLASS_NAMES if (train_df.label == c).any()]
    fig, axes = plt.subplots(len(picks), 4, figsize=(8, 2.2 * len(picks)))
    for i, r in enumerate(picks):
        with Image.open(r.path) as im:
            views = [plain(im)] + [aug(im) for _ in range(3)]
        for j, v in enumerate(views):
            axes[i][j].imshow(denormalize(v).permute(1, 2, 0).numpy())
            axes[i][j].axis("off")
            axes[i][j].set_title(f"{r.label}: " + ("original" if j == 0 else f"augmented {j}"), fontsize=7)
    plt.tight_layout()
    plt.savefig(cfg.out_dir / "augmentation_preview.png", dpi=150)
    plt.close()


# ----------------------------------------------------------------------------
# Task 6: PyTorch data loading
# ----------------------------------------------------------------------------
class BrainMRIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: T.Compose):
        self.paths = df.path.tolist()
        self.labels = [CLASS_TO_IDX[x] for x in df.label]
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, i: int):
        with Image.open(self.paths[i]) as im:
            x = self.transform(im)
        return x, self.labels[i]


def _seed_worker(worker_id: int) -> None:
    seed = torch.initial_seed() % 2 ** 32
    np.random.seed(seed)
    random.seed(seed)


def build_dataloaders(splits: Dict[str, pd.DataFrame], cfg: Config) -> Dict[str, DataLoader]:
    gen = torch.Generator()
    gen.manual_seed(cfg.seed)
    loaders = {}
    for name, part in splits.items():
        is_train = name == "train"
        loaders[name] = DataLoader(
            BrainMRIDataset(part, build_transforms(cfg.img_size, train=is_train)),
            batch_size=cfg.batch_size, shuffle=is_train, num_workers=cfg.num_workers,
            pin_memory=torch.cuda.is_available(), worker_init_fn=_seed_worker, generator=gen,
        )
    return loaders


# ----------------------------------------------------------------------------
# Task 7: integration
# ----------------------------------------------------------------------------
def run_pipeline(cfg: Config, copy_splits: bool = False, plots: bool = True):
    set_seed(cfg.seed)
    cfg.out_dir.mkdir(parents=True, exist_ok=True)
    before = dataset_fingerprint(cfg.data_root)

    df = inspect_dataset(collect_images(cfg.data_root))
    stats = explore(df, cfg.out_dir, plots)
    clean, removed, clean_report = clean_dataset(df, cfg.min_side)
    removed.drop(columns=["ok"], errors="ignore").to_csv(cfg.out_dir / "removed_files.csv", index=False)
    (cfg.out_dir / "cleaning_report.json").write_text(json.dumps(clean_report, indent=2))

    splits = split_dataset(clean, cfg)
    verify_splits(splits)
    for name, part in splits.items():
        part.to_csv(cfg.out_dir / f"{name}.csv", index=False)
    if plots:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        pd.DataFrame({k: v.label.value_counts() for k, v in splits.items()}).plot(kind="bar", figsize=(7, 4), title="Images per class in each split")
        plt.ylabel("Count"); plt.tight_layout(); plt.savefig(cfg.out_dir / "split_distribution.png", dpi=150); plt.close()
        save_augmentation_preview(splits["train"], cfg)
    if copy_splits:
        export_split_folders(splits, cfg.out_dir)

    loaders = build_dataloaders(splits, cfg)
    x, y = next(iter(loaders["train"]))
    log.info("Sample train batch: images %s %s, labels %s", tuple(x.shape), x.dtype, tuple(y.shape))

    if dataset_fingerprint(cfg.data_root) != before:
        raise AssertionError("The original dataset was modified")
    summary = {
        "analysis": stats, "cleaning": clean_report,
        "split_sizes": {k: len(v) for k, v in splits.items()},
        "split_fractions": {k: round(len(v) / len(clean), 4) for k, v in splits.items()},
        "batch_shape": list(x.shape), "original_dataset_unchanged": True,
    }
    (cfg.out_dir / "pipeline_summary.json").write_text(json.dumps(summary, indent=2))
    log.info("Split sizes: %s", summary["split_sizes"])
    return loaders, splits, summary


# ----------------------------------------------------------------------------
# Tests (run on a small synthetic dataset, so they work anywhere)
# ----------------------------------------------------------------------------
def make_synthetic_dataset(root: Path, per_class: int = 60) -> dict:
    """Random-noise images in many sizes and modes, plus a corrupt file, a duplicate and edge cases."""
    modes = ["L", "RGB", "RGBA", "P"]
    k = 0
    first = None
    for split in ["Training", "Testing"]:
        for cls in CLASS_NAMES:
            d = root / split / cls
            d.mkdir(parents=True, exist_ok=True)
            for i in range(per_class // 2):
                rs = np.random.RandomState(k)
                w, h = int(rs.randint(48, 140)), int(rs.randint(48, 140))
                im = Image.fromarray(rs.randint(0, 255, (h, w, 3), dtype=np.uint8)).convert(modes[k % 4])
                p = d / f"img_{k}.png"
                im.save(p)
                first = first or p
                k += 1
    special = {}
    ex = root / "Training" / "glioma"
    Image.fromarray(np.random.RandomState(9001).randint(0, 255, (32, 32), dtype=np.uint8)).save(ex / "edge_32.png")
    Image.fromarray(np.random.RandomState(9002).randint(0, 255, (31, 31), dtype=np.uint8)).save(ex / "edge_31.png")
    Image.fromarray(np.random.RandomState(9003).randint(0, 255, (20, 300), dtype=np.uint8)).save(ex / "extreme_aspect.png")
    (ex / "corrupt.jpg").write_bytes(b"\xff\xd8\xff\xe0 not a real image")
    (ex / "notes.txt").write_text("not an image")
    shutil.copy2(first, root / "Testing" / "glioma" / "duplicate_of_first.png")       # exact duplicate, same class
    other = root / "Training" / CLASS_NAMES[1] / f"img_{per_class // 2}.png"
    shutil.copy2(other, root / "Testing" / "pituitary" / "same_image_other_class.png")   # same image, different class
    special["corrupt"] = ex / "corrupt.jpg"
    return special


def run_tests() -> bool:
    """12 test cases. Returns True when all pass."""
    results = []

    def check(tid, what, expected, actual, ok):
        results.append((tid, what, expected, actual, "PASS" if ok else "FAIL"))

    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp) / "data"
        make_synthetic_dataset(root)
        cfg = Config(data_root=root, out_dir=Path(tmp) / "out", batch_size=8)
        fp_before = dataset_fingerprint(root)
        loaders, splits, summary = run_pipeline(cfg, plots=False)
        n = sum(len(v) for v in splits.values())

        frac = [len(splits[k]) / n for k in ["train", "val", "test"]]
        check("T1", "Split ratio", "70/15/15 within 1%", "/".join(f"{f*100:.1f}" for f in frac), all(abs(f - e) < 0.01 for f, e in zip(frac, [0.7, 0.15, 0.15])))

        try:
            verify_splits(splits); leak = 0
        except AssertionError:
            leak = 1
        check("T2", "No leakage between splits", "0 shared files or images", f"{leak} problems", leak == 0)

        total = pd.concat(splits.values())
        gap = max(abs(int((splits[k].label == c).sum()) - int((total.label == c).sum()) * f)
                  for c in CLASS_NAMES for k, f in [("train", cfg.train_frac), ("val", cfg.val_frac), ("test", cfg.test_frac)])
        check("T3", "Stratified class balance", "each class within 1 image of its share", f"largest gap {gap:.1f} image(s)", gap <= 1.0)

        x, y = next(iter(loaders["train"]))
        check("T4", "Batch shape and type", "(8,3,224,224) float32, labels 0-3", f"{tuple(x.shape)} {x.dtype}, labels {int(y.min())}-{int(y.max())}",
              tuple(x.shape) == (8, 3, 224, 224) and x.dtype == torch.float32 and 0 <= int(y.min()) and int(y.max()) <= 3)

        sample = Image.open(splits["train"].path[0]).convert("RGB")
        ev, tr = build_transforms(224, False), build_transforms(224, True)
        same_eval = torch.equal(ev(sample), ev(sample))
        torch.manual_seed(0)
        differs = any(not torch.equal(tr(sample), tr(sample)) for _ in range(5))
        check("T5", "Augmentation only on training", "val/test repeatable, train varies", f"eval same={same_eval}, train varies={differs}", same_eval and differs)

        ok_modes = True
        for mode in ["L", "RGBA", "P", "RGB"]:
            im = Image.new("RGB", (40, 40), (10, 20, 30)).convert(mode)
            ok_modes &= tuple(ev(im).shape) == (3, 224, 224)
        wide = Image.new("L", (300, 20), 128)
        ok_modes &= tuple(ev(wide).shape) == (3, 224, 224)
        check("T6", "(boundary) mixed modes and extreme aspect ratio", "all become (3,224,224)", "all (3,224,224)" if ok_modes else "shape mismatch", ok_modes)

        kept = set(Path(p).name for p in pd.concat(splits.values()).path)
        check("T7", "(boundary) minimum size rule", "32x32 kept, 31x31 removed", f"32 kept={'edge_32.png' in kept}, 31 kept={'edge_31.png' in kept}", "edge_32.png" in kept and "edge_31.png" not in kept)

        check("T8", "(negative) corrupt and non-image files", "1 corrupt file reported, txt ignored", f"{summary['cleaning']['corrupt_or_unreadable']} corrupt reported",
              summary["cleaning"]["corrupt_or_unreadable"] == 1 and "corrupt.jpg" not in kept)

        check("T9", "(negative) duplicate image", "1 duplicate removed", f"{summary['cleaning']['exact_duplicates_removed']} removed", summary["cleaning"]["exact_duplicates_removed"] == 1)

        check("T10", "Original dataset preserved", "fingerprint unchanged", "unchanged" if dataset_fingerprint(root) == fp_before else "CHANGED", dataset_fingerprint(root) == fp_before)

        again = split_dataset(clean_dataset(inspect_dataset(collect_images(root)), cfg.min_side)[0], cfg)
        same = all(again[k].path.tolist() == splits[k].path.tolist() for k in splits)
        check("T11", "Reproducibility with fixed seed", "identical splits on re-run", "identical" if same else "different", same)

        check("T12", "(negative) same image in two classes", "both copies removed (2 conflicts)", f"{summary['cleaning']['label_conflicts']} removed", summary["cleaning"]["label_conflicts"] == 2)

    print("\n{:<5}{:<48}{:<38}{:<42}{}".format("ID", "Test", "Expected", "Actual", "Result"))
    for r in results:
        print("{:<5}{:<48}{:<38}{:<42}{}".format(*r))
    passed = sum(r[4] == "PASS" for r in results)
    print(f"\nTotal: {len(results)}  Passed: {passed}  Failed: {len(results) - passed}")
    return passed == len(results)


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="Brain tumor MRI preprocessing pipeline")
    ap.add_argument("--data", type=Path, help="dataset folder that contains Training and Testing")
    ap.add_argument("--out", type=Path, default=Path("outputs"))
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--num-workers", type=int, default=0)
    ap.add_argument("--copy-splits", action="store_true", help="also copy images into organized/train|val|test/<class>")
    ap.add_argument("--no-plots", action="store_true")
    ap.add_argument("--test", action="store_true", help="run the test cases and exit")
    args = ap.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")
    if args.test:
        sys.exit(0 if run_tests() else 1)
    if not args.data:
        ap.error("--data is required (or use --test)")
    run_pipeline(Config(args.data, args.out, args.img_size, batch_size=args.batch_size, num_workers=args.num_workers),
                 copy_splits=args.copy_splits, plots=not args.no_plots)


Overwriting brain_tumor_preprocessing_pipeline.py


In [25]:
!python brain_tumor_preprocessing_pipeline.py --test

11:55:10 INFO Collected 246 images in 4 classes
11:55:10 INFO Exploration done: 240 unique resolutions, modes {'L': 64, 'RGBA': 61, 'RGB': 60, 'P': 60}
11:55:10 INFO Cleaning: {'images_before_cleaning': 246, 'corrupt_or_unreadable': 1, 'too_small': 2, 'label_conflicts': 2, 'exact_duplicates_removed': 1, 'images_after_cleaning': 240, 'class_counts_after_cleaning': {'glioma': 61, 'notumor': 60, 'pituitary': 60, 'meningioma': 59}}
11:55:10 INFO Sample train batch: images (8, 3, 224, 224) torch.float32, labels (8,)
11:55:10 INFO Split sizes: {'train': 167, 'val': 36, 'test': 37}
11:55:11 INFO Collected 246 images in 4 classes
11:55:11 INFO Cleaning: {'images_before_cleaning': 246, 'corrupt_or_unreadable': 1, 'too_small': 2, 'label_conflicts': 2, 'exact_duplicates_removed': 1, 'images_after_cleaning': 240, 'class_counts_after_cleaning': {'glioma': 61, 'notumor': 60, 'pituitary': 60, 'meningioma': 59}}

ID   Test                                            Expected                            

In [27]:
%%writefile baseline_demo.py
"""
Baseline demo: fine-tune a pretrained CNN on the preprocessed MRI data, evaluate it,
and predict one scan with its confidence.

Put this file in the same folder as brain_tumor_preprocessing_pipeline.py.

  Quick test (2 minutes, checks everything works):
    python baseline_demo.py train --data "C:/path/to/dataset" --epochs 1 --limit 5
  Real run (use a GPU, e.g. a Kaggle notebook with GPU turned on):
    python baseline_demo.py train --data /kaggle/input/<dataset> --epochs 3 --num-workers 2
  Predict one scan:
    python baseline_demo.py predict --image "C:/path/to/scan.jpg"

The first run downloads pretrained weights, so internet must be on.
"""
from __future__ import annotations

import argparse
import json
import logging
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torchvision import models

from brain_tumor_preprocessing_pipeline import CLASS_NAMES, Config, build_transforms, run_pipeline, set_seed

log = logging.getLogger("baseline")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(name: str) -> nn.Module:
    n = len(CLASS_NAMES)
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, n)
    else:
        raise ValueError("model must be resnet50, efficientnet_b0 or densenet121")
    return m.to(DEVICE)


def run_epoch(model, loader, loss_fn, optimizer=None, limit=0):
    """One pass over the loader. Trains if an optimizer is given, otherwise only evaluates."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count, preds, labels = 0.0, 0, 0, [], []
    with torch.set_grad_enabled(training):
        for i, (x, y) in enumerate(loader):
            if limit and i >= limit:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = loss_fn(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            p = out.argmax(1)
            correct += (p == y).sum().item()
            count += len(y)
            preds += p.cpu().tolist()
            labels += y.cpu().tolist()
    return total_loss / count, correct / count, preds, labels


def save_test_report(name, test_acc, preds, labels, out_dir, results_file, cm_file):
    """Print and save accuracy, the per-class report and the confusion matrix."""
    idx = list(range(len(CLASS_NAMES)))
    report = classification_report(labels, preds, labels=idx, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=idx)
    (Path(out_dir) / results_file).write_text(json.dumps({"model": name, "test_accuracy": test_acc, "report": report}, indent=2))
    print(f"\nTest accuracy: {test_acc:.4f}")
    print(classification_report(labels, preds, labels=idx, target_names=CLASS_NAMES, zero_division=0))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(idx); ax.set_xticklabels(CLASS_NAMES, rotation=30); ax.set_yticks(idx); ax.set_yticklabels(CLASS_NAMES)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, v, ha="center", va="center")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix ({name})")
    plt.tight_layout(); plt.savefig(Path(out_dir) / cm_file, dpi=150); plt.close()


def train(args):
    set_seed(42)
    cfg = Config(Path(args.data), Path(args.out), batch_size=args.batch_size, num_workers=args.num_workers)
    loaders, _, _ = run_pipeline(cfg, plots=False)          # reuses the cleaned, leak-free split
    model = build_model(args.model)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    best_acc, ckpt = 0.0, cfg.out_dir / "baseline.pt"
    for epoch in range(1, args.epochs + 1):
        tl, ta, _, _ = run_epoch(model, loaders["train"], loss_fn, optimizer, args.limit)
        vl, va, _, _ = run_epoch(model, loaders["val"], loss_fn, None, args.limit)
        log.info("epoch %d  train loss %.3f acc %.3f | val loss %.3f acc %.3f", epoch, tl, ta, vl, va)
        if va >= best_acc:                                   # keep the best model on the validation set
            best_acc = va
            torch.save({"model": model.state_dict(), "name": args.model, "img_size": cfg.img_size}, ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model"])
    _, test_acc, preds, labels = run_epoch(model, loaders["test"], loss_fn, None, args.limit)   # test set used once, at the end
    save_test_report(args.model, test_acc, preds, labels, cfg.out_dir, "baseline_results.json", "confusion_matrix.png")
    log.info("Saved model, results and confusion matrix in %s", cfg.out_dir)


def predict(args):
    ckpt = torch.load(Path(args.out) / "baseline.pt", map_location=DEVICE)
    model = build_model(ckpt["name"])
    model.load_state_dict(ckpt["model"])
    model.eval()
    with Image.open(args.image) as im:
        x = build_transforms(ckpt["img_size"], train=False)(im).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0].cpu()
    top = int(probs.argmax())
    print(f"\nPrediction: {CLASS_NAMES[top]}  (confidence {probs[top]*100:.1f}%)")
    for name, p in sorted(zip(CLASS_NAMES, probs.tolist()), key=lambda t: -t[1]):
        print(f"  {name:<12}{p*100:5.1f}%")
    print("\nNote: this is a student baseline model, not a medical diagnosis.")


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")
    ap = argparse.ArgumentParser(description="Baseline training and prediction demo")
    sub = ap.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("train")
    t.add_argument("--data", required=True, help="dataset folder with Training and Testing")
    t.add_argument("--out", default="outputs")
    t.add_argument("--model", default="resnet50", choices=["resnet50", "efficientnet_b0", "densenet121"])
    t.add_argument("--epochs", type=int, default=3)
    t.add_argument("--batch-size", type=int, default=32)
    t.add_argument("--num-workers", type=int, default=0)
    t.add_argument("--lr", type=float, default=1e-4)
    t.add_argument("--limit", type=int, default=0, help="batches per epoch, for a quick test (0 = all)")
    p = sub.add_parser("predict")
    p.add_argument("--image", required=True)
    p.add_argument("--out", default="outputs")
    a = ap.parse_args()
    train(a) if a.cmd == "train" else predict(a)

Overwriting baseline_demo.py


In [3]:
import os
print(os.listdir("/kaggle/input/datasets/masoudnickparvar"))

['brain-tumor-mri-dataset']


In [28]:
import glob, os, subprocess, json
import pandas as pd

hits = glob.glob('/kaggle/input/**/Training', recursive=True)
DATA = os.path.dirname(hits[0])
print("Dataset folder:", DATA)

def run(args):
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()

Dataset folder: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset


In [11]:
run(["python", "/kaggle/working/baseline_demo.py", "train", "--data", DATA, "--out", "/kaggle/working/outputs_effnet", "--model", "efficientnet_b0", "--epochs", "3", "--num-workers", "2"])

07:40:13 INFO Collected 7200 images in 4 classes
07:41:10 INFO Inspected 1000 / 7200 images
07:42:06 INFO Inspected 2000 / 7200 images
07:43:00 INFO Inspected 3000 / 7200 images
07:44:00 INFO Inspected 4000 / 7200 images
07:44:50 INFO Inspected 5000 / 7200 images
07:45:38 INFO Inspected 6000 / 7200 images
07:46:41 INFO Inspected 7000 / 7200 images
07:46:54 INFO Exploration done: 447 unique resolutions, modes {'RGB': 4129, 'L': 3067, 'RGBA': 3, 'P': 1}
07:46:54 INFO Cleaning: {'images_before_cleaning': 7200, 'corrupt_or_unreadable': 0, 'too_small': 0, 'label_conflicts': 0, 'exact_duplicates_removed': 187, 'images_after_cleaning': 7013, 'class_counts_after_cleaning': {'glioma': 1786, 'meningioma': 1784, 'pituitary': 1762, 'notumor': 1681}}
07:46:55 INFO Sample train batch: images (32, 3, 224, 224) torch.float32, labels (32,)
07:46:58 INFO Split sizes: {'train': 4909, 'val': 1052, 'test': 1052}
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /r

In [ ]:
run(["python", "/kaggle/working/baseline_demo.py", "train", "--data", DATA, "--out", "/kaggle/working/outputs_densenet", "--model", "densenet121", "--epochs", "3", "--num-workers", "2"])

In [ ]:
run(["python", "/kaggle/working/baseline_demo.py", "train", "--data", DATA, "--out", "/kaggle/working/outputs_resnet", "--model", "resnet50", "--epochs", "3", "--num-workers", "2"])

In [ ]:
rows = []
for folder in ["outputs", "outputs_effnet", "outputs_densenet","outputs_resnet"]:
    p = f"/kaggle/working/{folder}/baseline_results.json"
    if os.path.exists(p):
        r = json.load(open(p))
        rows.append({"model": r["model"], "test_accuracy": round(r["test_accuracy"], 4), "macro_f1": round(r["report"]["macro avg"]["f1-score"], 4)})
pd.DataFrame(rows)

In [39]:
%%writefile dualbranch_demo.py
"""
Dual-branch attention model: two pretrained backbones, a CBAM attention block on each,
features joined together, and one classifier.

Put this file in the same folder as brain_tumor_preprocessing_pipeline.py and baseline_demo.py.

  Quick check (a few minutes):
    python dual_branch_demo.py train --data /path/to/dataset --out outputs_dual --epochs 1 --limit 5
  Real run:
    python dual_branch_demo.py train --data /path/to/dataset --out outputs_dual --epochs 3 --num-workers 2
  Options for the later ablation study:
    --no-cbam              removes the attention blocks
    --branch-b none        uses only one backbone
"""
from __future__ import annotations

import argparse
import logging
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models

from baseline_demo import DEVICE, run_epoch, save_test_report
from brain_tumor_preprocessing_pipeline import CLASS_NAMES, Config, run_pipeline, set_seed

log = logging.getLogger("dual")


class ChannelAttention(nn.Module):
    """Learns WHICH feature channels matter."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(channels, channels // reduction), nn.ReLU(inplace=True), nn.Linear(channels // reduction, channels))

    def forward(self, x):
        avg = self.mlp(x.mean(dim=(2, 3)))
        mx = self.mlp(x.amax(dim=(2, 3)))
        return x * torch.sigmoid(avg + mx)[:, :, None, None]


class SpatialAttention(nn.Module):
    """Learns WHERE in the image to look."""
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2)

    def forward(self, x):
        pooled = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * torch.sigmoid(self.conv(pooled))


class CBAM(nn.Module):
    """Convolutional Block Attention Module: channel attention, then spatial attention."""
    def __init__(self, channels: int):
        super().__init__()
        self.channel = ChannelAttention(channels)
        self.spatial = SpatialAttention()

    def forward(self, x):
        return self.spatial(self.channel(x))


def make_branch(name: str):
    """Returns (feature extractor, number of output channels) for a pretrained backbone."""
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        return nn.Sequential(*list(m.children())[:-2]), 2048
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        return m.features, 1280
    if name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        return nn.Sequential(m.features, nn.ReLU(inplace=True)), 1024
    raise ValueError("backbone must be resnet50, efficientnet_b0 or densenet121")


class DualBranchModel(nn.Module):
    def __init__(self, branch_a="resnet50", branch_b="efficientnet_b0", use_cbam=True, dropout=0.3, num_classes=4):
        super().__init__()
        self.body_a, ch_a = make_branch(branch_a)
        self.cbam_a = CBAM(ch_a) if use_cbam else nn.Identity()
        total = ch_a
        self.body_b = self.cbam_b = None
        if branch_b and branch_b != "none":
            self.body_b, ch_b = make_branch(branch_b)
            self.cbam_b = CBAM(ch_b) if use_cbam else nn.Identity()
            total += ch_b
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)          # the same dropout is reused later for Monte Carlo Dropout
        self.fc = nn.Linear(total, num_classes)

    def forward(self, x):
        feats = [self.pool(self.cbam_a(self.body_a(x))).flatten(1)]
        if self.body_b is not None:
            feats.append(self.pool(self.cbam_b(self.body_b(x))).flatten(1))
        return self.fc(self.drop(torch.cat(feats, dim=1)))


def train(args):
    set_seed(42)
    cfg = Config(Path(args.data), Path(args.out), batch_size=args.batch_size, num_workers=args.num_workers)
    loaders, _, _ = run_pipeline(cfg, plots=False)             # same cleaned, leak-free split as the baselines
    model = DualBranchModel(args.branch_a, args.branch_b, use_cbam=not args.no_cbam, dropout=args.dropout, num_classes=len(CLASS_NAMES)).to(DEVICE)
    name = f"dual_{args.branch_a}_{args.branch_b}" + ("_nocbam" if args.no_cbam else "")
    log.info("Model %s, %d parameters", name, sum(p.numel() for p in model.parameters()))
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    best_acc, ckpt = 0.0, cfg.out_dir / "dual_branch.pt"
    for epoch in range(1, args.epochs + 1):
        tl, ta, _, _ = run_epoch(model, loaders["train"], loss_fn, optimizer, args.limit)
        vl, va, _, _ = run_epoch(model, loaders["val"], loss_fn, None, args.limit)
        log.info("epoch %d  train loss %.3f acc %.3f | val loss %.3f acc %.3f", epoch, tl, ta, vl, va)
        if va >= best_acc:
            best_acc = va
            torch.save({"model": model.state_dict(), "name": name, "args": vars(args), "img_size": cfg.img_size}, ckpt)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model"])
    _, test_acc, preds, labels = run_epoch(model, loaders["test"], loss_fn, None, args.limit)
    save_test_report(name, test_acc, preds, labels, cfg.out_dir, "dual_results.json", "confusion_matrix_dual.png")


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")
    ap = argparse.ArgumentParser(description="Dual-branch CBAM model")
    sub = ap.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("train")
    t.add_argument("--data", required=True)
    t.add_argument("--out", default="outputs_dual")
    t.add_argument("--branch-a", default="resnet50", choices=["resnet50", "efficientnet_b0", "densenet121"])
    t.add_argument("--branch-b", default="efficientnet_b0", choices=["resnet50", "efficientnet_b0", "densenet121", "none"])
    t.add_argument("--no-cbam", action="store_true", help="remove attention (for the ablation study)")
    t.add_argument("--dropout", type=float, default=0.3)
    t.add_argument("--epochs", type=int, default=3)
    t.add_argument("--batch-size", type=int, default=32)
    t.add_argument("--num-workers", type=int, default=0)
    t.add_argument("--lr", type=float, default=1e-4)
    t.add_argument("--limit", type=int, default=0, help="batches per epoch, for a quick test (0 = all)")
    train(ap.parse_args())

Overwriting dualbranch_demo.py


In [41]:
run(["python", "dualbranch_demo.py", "train", "--data", DATA, "--out", "outputs_dual", "--epochs", "3", "--num-workers", "2"])

12:00:11 INFO Collected 7200 images in 4 classes
12:00:13 INFO Inspected 1000 / 7200 images
12:00:15 INFO Inspected 2000 / 7200 images
12:00:18 INFO Inspected 3000 / 7200 images
12:00:20 INFO Inspected 4000 / 7200 images
12:00:23 INFO Inspected 5000 / 7200 images
12:00:25 INFO Inspected 6000 / 7200 images
12:00:28 INFO Inspected 7000 / 7200 images
12:00:29 INFO Exploration done: 447 unique resolutions, modes {'RGB': 4129, 'L': 3067, 'RGBA': 3, 'P': 1}
12:00:29 INFO Cleaning: {'images_before_cleaning': 7200, 'corrupt_or_unreadable': 0, 'too_small': 0, 'label_conflicts': 0, 'exact_duplicates_removed': 187, 'images_after_cleaning': 7013, 'class_counts_after_cleaning': {'glioma': 1786, 'meningioma': 1784, 'pituitary': 1762, 'notumor': 1681}}
12:00:30 INFO Sample train batch: images (32, 3, 224, 224) torch.float32, labels (32,)
12:00:33 INFO Split sizes: {'train': 4909, 'val': 1052, 'test': 1052}
12:00:34 INFO Model dual_resnet50_efficientnet_b0, 28261718 parameters
12:01:50 INFO epoch 1  t